In [ ]:
import jax
import jax.numpy as jnp
from functools import partial
from jax.test_util import check_grads


from py2d.Py2D_solver import Py2D_solver


# -- 1) Problem setup (constants) --
Re, fkx, fky       = 20e3, 4, 4
alpha, beta         = 0.1, 0
dt              = 1e-3
NX_high_res        = 32
resumeSimulation   = False

# -- 2) Freeze all the static / non-differentiable args --
test_solver = partial(
    Py2D_solver,
    Re=Re,
    fkx=fkx,
    fky=fky,
    alpha=alpha,
    beta=beta,
    NX=NX_high_res,
    saveData=False,          # no file I/O during tests
    SGSModel_string='LEITH',
    dt=dt,
    tSAVE=dt,
    tTotal=dt,
    readTrue=False,
    ICnum=1,
    direct_IC=None,
    resumeSim=resumeSimulation,
)

# -- 3) Build a scalar “loss” exposing the float params --
def loss_fn(eddyVisc, errorTerm):
    omega, *_ = test_solver(
        eddyViscosityCoeff=eddyVisc,
        error_term_hat=errorTerm,
    )
    # Ensure omega is a JAX array and collapse the array output to a single scalar
    return jnp.sum(jnp.abs(jnp.asarray(omega)))

# -- 4) Pick test-points for each param --

eddyVisc0             = 0.17
errorTerm0            = 0.1

# -- 5) Smoke-test both forward & reverse autodiff up to 2nd order --
check_grads(
    loss_fn,
    (eddyVisc0, errorTerm0,),  # Wrap errorTerm0 in a tuple
    order=2,
    modes=['fwd', 'rev']
)

print("✅ gradients OK for eddyViscosityCoeff and error_term_hat!")


gpu
[CudaDevice(id=0)]
